# SteelVision — segmentação de defeitos em aço
**Flávia Gaia · PyTorch · U-Net · Visão Computacional industrial**

Este notebook implementa exploração, preparação, treino, avaliação por classe e análise de erros.
Não contém resultados pré-calculados em dados Severstal.

**Para executar no Kaggle:** aceite as regras da competição Severstal, adicione seus dados ao notebook
e selecione GPU nas configurações, se disponível. O pacote está incorporado abaixo; não é necessário
clonar repositório, baixar pesos ou ativar internet. Bibliotecas esperadas: PyTorch, NumPy, Pillow e Matplotlib.

O teste reservado será uma divisão do conjunto público anotado de treino. Os dados oficiais sem rótulos
da competição não são usados para calcular qualidade. A primeira execução usa duas épocas para verificar
o pipeline, não para prometer convergência ou um resultado competitivo.

[Severstal](https://www.kaggle.com/competitions/severstal-steel-defect-detection/data) ·
[U-Net](https://arxiv.org/abs/1505.04597)


## 1. Configuração
`USE_SYNTHETIC = False` usa Severstal. Altere para `True` apenas para testar o fluxo sem a competição.
Resultados sintéticos são identificados e não medem qualidade industrial.

Para repetir, altere `RUN_NAME`. Não sobrescreva uma execução. Aumente `EPOCHS` somente após validar o fluxo.
O limiar é fixo em 0,5. Não use resultados do teste para ajustar modelos ou limiares.


In [ ]:
from pathlib import Path

USE_SYNTHETIC = False
EPOCHS = 2
RUN_NAME = "baseline_01"
SEED = 42
BATCH_SIZE = 4
DATA_DIR_OVERRIDE = None  # opcional: Path("/caminho/para/severstal")

BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT = BASE / "steelvision_artifacts" / RUN_NAME
if OUTPUT.exists():
    raise FileExistsError(f"A execução {OUTPUT} já existe. Escolha outro RUN_NAME.")
PACKAGE_ROOT = BASE / "steelvision_notebook_source"
PACKAGE = PACKAGE_ROOT / "steelvision"
PACKAGE.mkdir(parents=True, exist_ok=True)


## 2. Código do projeto
As próximas células gravam o pacote localmente. O mesmo código é mantido nos módulos do repositório.
Você pode examinar as implementações de RLE, split, U-Net, métricas e treinamento aqui.


### `__init__.py`

In [ ]:
source = r'''"""SteelVision: baseline de segmentação multirrótulo de defeitos em aço."""

__version__ = "0.1.0"
'''
(PACKAGE / '__init__.py').write_text(source, encoding='utf-8')

### `cli.py`

In [ ]:
source = r'''"""Interface de linha de comando para o projeto de inspeção de aço."""

import argparse
import json
from pathlib import Path


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    prepare = commands.add_parser("prepare", help="Validar RLE e dividir imagens por hash")
    prepare.add_argument("--data", required=True)
    prepare.add_argument("--out", required=True)
    prepare.add_argument("--seed", type=int, default=42)
    eda = commands.add_parser("eda", help="Exploração das classes e máscaras")
    eda.add_argument("--data", required=True)
    eda.add_argument("--manifest", required=True)
    eda.add_argument("--out", required=True)
    train = commands.add_parser("train", help="Treinar U-Net e selecionar na validação")
    train.add_argument("--data", required=True)
    train.add_argument("--manifest", required=True)
    train.add_argument("--out", required=True)
    train.add_argument("--epochs", type=int, default=10)
    train.add_argument("--batch-size", type=int, default=4)
    train.add_argument("--height", type=int, default=128)
    train.add_argument("--width", type=int, default=800)
    train.add_argument("--base-channels", type=int, default=16)
    train.add_argument("--seed", type=int, default=42)
    train.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda", "mps"])
    train.add_argument("--num-workers", type=int, default=0)
    evaluate = commands.add_parser("evaluate", help="Avaliar no teste reservado")
    evaluate.add_argument("--data", required=True)
    evaluate.add_argument("--manifest", required=True)
    evaluate.add_argument("--checkpoint", required=True)
    evaluate.add_argument("--out", required=True)
    evaluate.add_argument("--device", default="auto")
    predict = commands.add_parser("predict", help="Inferência em uma imagem")
    predict.add_argument("--image", required=True)
    predict.add_argument("--checkpoint", required=True)
    predict.add_argument("--out", required=True)
    predict.add_argument("--device", default="auto")
    demo = commands.add_parser("demo", help="Executar fluxo completo com dados SINTÉTICOS")
    demo.add_argument("--out", default="artifacts/synthetic_demo")
    demo.add_argument("--epochs", type=int, default=2)
    args = parser.parse_args(argv)
    if args.command == "prepare":
        from .data import prepare_manifest

        print(prepare_manifest(args.data, args.out, args.seed))
    elif args.command == "eda":
        from .visualize import explore

        print(json.dumps(explore(args.data, args.manifest, args.out), indent=2))
    elif args.command == "train":
        from .engine import Config, train

        config = Config(
            epochs=args.epochs,
            batch_size=args.batch_size,
            height=args.height,
            width=args.width,
            base_channels=args.base_channels,
            seed=args.seed,
            device=args.device,
            num_workers=args.num_workers,
            data_kind="synthetic" if (Path(args.data) / "SYNTHETIC.txt").exists() else "severstal",
        )
        print(train(args.data, args.manifest, args.out, config))
    elif args.command == "evaluate":
        from .engine import evaluate

        print(
            json.dumps(
                evaluate(args.data, args.manifest, args.checkpoint, args.out, device=args.device),
                indent=2,
            )
        )
    elif args.command == "predict":
        from .engine import predict

        result = predict(args.image, args.checkpoint, args.out, args.device)
        print(f"Predição salva: {result['image_id']}")
    elif args.command == "demo":
        import torch

        from .data import prepare_manifest
        from .engine import Config, evaluate, predict, train
        from .synthetic import generate_synthetic
        from .visualize import explore

        # Smoke test pequeno e previsível mesmo sem acelerador.
        torch.set_num_threads(2)
        root = Path(args.out)
        data = generate_synthetic(root / "data")
        manifest = prepare_manifest(data, root / "prepared")
        explore(data, manifest, root / "eda", size=(64, 256))
        config = Config(
            epochs=args.epochs,
            height=64,
            width=256,
            base_channels=8,
            device="cpu",
            data_kind="synthetic",
            batch_size=4,
        )
        checkpoint = train(data, manifest, root / "run", config)
        evaluate(data, manifest, checkpoint, root / "evaluation", device="cpu")
        predict(
            next((data / "train_images").glob("*.png")),
            checkpoint,
            root / "inference",
            device="cpu",
        )
        print(f"DEMO SINTÉTICA concluída: {root.resolve()}")


if __name__ == "__main__":
    main()
'''
(PACKAGE / 'cli.py').write_text(source, encoding='utf-8')

### `data.py`

In [ ]:
source = r'''"""Leitura Severstal, RLE por coluna e divisão antes de qualquer transformação."""

import csv
import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageOps
from torch.utils.data import Dataset

CLASSES = (1, 2, 3, 4)


def decode_rle(value, height, width):
    """Decodifica RLE Severstal: índices a partir de 1, ordem Fortran (colunas)."""
    mask = np.zeros(height * width, dtype=np.uint8)
    if value is None or not str(value).strip():
        return mask.reshape((height, width), order="F")
    try:
        runs = np.asarray(str(value).split(), dtype=np.int64)
    except ValueError as exc:
        raise ValueError("RLE deve conter inteiros") from exc
    if len(runs) % 2:
        raise ValueError("RLE deve conter pares início/comprimento")
    starts, lengths = runs[0::2] - 1, runs[1::2]
    if np.any(starts < 0) or np.any(lengths <= 0) or np.any(starts + lengths > mask.size):
        raise ValueError("RLE fora dos limites da imagem")
    for start, length in zip(starts, lengths):
        mask[start : start + length] = 1
    return mask.reshape((height, width), order="F")


def encode_rle(mask):
    """Inverso de decode_rle; usado também na demonstração sintética."""
    flat = (np.asarray(mask).ravel(order="F") > 0).astype(np.uint8)
    padded = np.concatenate([[0], flat, [0]])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    changes[1::2] -= changes[0::2]
    return " ".join(map(str, changes))


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_annotations(csv_path):
    """Aceita ImageId/ClassId e o formato legado ImageId_ClassId."""
    annotations = {}
    with Path(csv_path).open(newline="", encoding="utf-8-sig") as stream:
        for row in csv.DictReader(stream):
            if "ImageId_ClassId" in row:
                image_id, class_id = row["ImageId_ClassId"].rsplit("_", 1)
            else:
                image_id, class_id = row["ImageId"], row["ClassId"]
            class_id = int(class_id)
            if class_id not in CLASSES:
                raise ValueError(f"Classe desconhecida: {class_id}")
            key = (image_id, class_id)
            if key in annotations:
                raise ValueError(f"Anotação duplicada: {key}")
            annotations[key] = (row.get("EncodedPixels") or "").strip()
    return annotations


def prepare_manifest(data_dir, output_dir, seed=42):
    """Divide imagens por grupos de hash, incluindo imagens sem defeitos."""
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    image_dir = data_dir / "train_images"
    annotations = read_annotations(data_dir / "train.csv")
    paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {".jpg", ".png", ".jpeg"})
    if len(paths) < 3:
        raise ValueError("É necessário ter pelo menos três imagens")
    missing = {name for name, _ in annotations} - {p.name for p in paths}
    if missing:
        raise ValueError(f"Anotações sem imagem: {sorted(missing)[:5]}")
    records, groups = [], {}
    for path in paths:
        with Image.open(path) as image:
            width, height = image.size
            image.verify()
        rles = [annotations.get((path.name, c), "") for c in CLASSES]
        # Validar os limites antes de iniciar um treinamento demorado.
        for rle in rles:
            decode_rle(rle, height, width)
        record = {
            "image_id": path.name,
            "sha256": sha256_file(path),
            "height": height,
            "width": width,
            "rles": rles,
        }
        records.append(record)
        groups.setdefault(record["sha256"], []).append(record)
    hashes = sorted(groups)
    if len(hashes) < 3:
        raise ValueError("É necessário ter três grupos de imagens distintas")
    # Conteúdo idêntico com anotações incompatíveis deve ser revisado.
    for group in groups.values():
        if len({tuple(r["rles"]) for r in group}) > 1:
            raise ValueError("Imagens idênticas possuem anotações conflitantes")
    random.Random(seed).shuffle(hashes)
    n_test = max(1, round(len(hashes) * 0.15))
    n_val = max(1, round(len(hashes) * 0.15))
    assignments = {}
    for i, digest in enumerate(hashes):
        assignments[digest] = "test" if i < n_test else "val" if i < n_test + n_val else "train"
    for record in records:
        record["split"] = assignments[record["sha256"]]
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        "seed": seed,
        "split_method": "seeded_exact_file_hash_groups_70_15_15",
        "csv_sha256": sha256_file(data_dir / "train.csv"),
        "records": records,
    }
    target = output_dir / "manifest.json"
    target.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return target


class SteelDataset(Dataset):
    """Máscaras independentes por classe; aumentos sincronizados só no treino."""

    def __init__(self, data_dir, manifest_path, split, size=(128, 800), augment=False):
        self.image_dir = Path(data_dir) / "train_images"
        manifest = json.loads(Path(manifest_path).read_text())
        self.records = [r for r in manifest["records"] if r["split"] == split]
        self.size = tuple(size)
        self.augment = augment
        if not self.records:
            raise ValueError(f"Divisão vazia: {split}")
        if augment and split != "train":
            raise ValueError("Aumentos permitidos somente no treino")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        with Image.open(self.image_dir / record["image_id"]) as original:
            image = original.convert("RGB")
        if image.size != (record["width"], record["height"]):
            raise ValueError("Dimensões alteradas após a criação do manifesto")
        masks = [
            Image.fromarray(decode_rle(rle, record["height"], record["width"]))
            for rle in record["rles"]
        ]
        if self.augment:
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
                masks = [ImageOps.mirror(m) for m in masks]
            if random.random() < 0.5:
                image = ImageOps.flip(image)
                masks = [ImageOps.flip(m) for m in masks]
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
        target_size = (self.size[1], self.size[0])
        image = image.resize(target_size, Image.Resampling.BILINEAR)
        x = np.asarray(image, dtype=np.float32).transpose(2, 0, 1) / 255.0
        y = np.stack([np.asarray(m.resize(target_size, Image.Resampling.NEAREST)) for m in masks])
        return (
            torch.from_numpy(x.copy()),
            torch.from_numpy(y.astype(np.float32)),
            record["image_id"],
        )
'''
(PACKAGE / 'data.py').write_text(source, encoding='utf-8')

### `engine.py`

In [ ]:
source = r'''"""Treinamento, seleção por validação e avaliação única no teste reservado."""

import csv
import json
import platform
import random
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from .data import SteelDataset, sha256_file
from .metrics import SegmentationMetrics
from .model import UNet, segmentation_loss
from .visualize import plot_history, save_comparison


@dataclass
class Config:
    epochs: int = 10
    batch_size: int = 4
    height: int = 128
    width: int = 800
    base_channels: int = 16
    learning_rate: float = 1e-3
    seed: int = 42
    num_workers: int = 0
    patience: int = 4
    threshold: float = 0.5
    device: str = "auto"
    data_kind: str = "severstal"


def choose_device(value="auto"):
    if value != "auto":
        return torch.device(value)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def loader(data_dir, manifest, split, config, shuffle=False):
    dataset = SteelDataset(
        data_dir, manifest, split, (config.height, config.width), augment=shuffle
    )
    generator = torch.Generator().manual_seed(config.seed)
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=shuffle,
        num_workers=config.num_workers,
        worker_init_fn=seed_worker,
        generator=generator,
        pin_memory=config.device == "cuda",
    )


@torch.inference_mode()
def evaluate_loader(model, batches, device, threshold):
    model.eval()
    meter = SegmentationMetrics()
    loss_sum = count = 0
    for image, target, names in batches:
        image, target = image.to(device), target.to(device)
        logits = model(image)
        loss_sum += float(segmentation_loss(logits, target)) * len(names)
        count += len(names)
        meter.update(logits.sigmoid().cpu().numpy(), target.cpu().numpy(), names, threshold)
    summary = meter.summary()
    summary["loss"] = loss_sum / count
    return summary, meter.rows


def git_revision():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def train(data_dir, manifest, output_dir, config):
    if config.epochs < 1 or config.patience < 1:
        raise ValueError("epochs e patience devem ser positivos")
    if config.height < 8 or config.width < 8 or not 0 < config.threshold < 1:
        raise ValueError("Dimensões >= 8 e limiar entre 0 e 1 são obrigatórios")
    output_dir = Path(output_dir)
    if (output_dir / "best.pt").exists():
        raise FileExistsError("Use outro diretório para preservar a execução anterior")
    output_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(config.seed)
    device = choose_device(config.device)
    model = UNet(config.base_channels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    train_batches = loader(data_dir, manifest, "train", config, shuffle=True)
    val_batches = loader(data_dir, manifest, "val", config)
    metadata = {
        "config": asdict(config),
        "device": str(device),
        "torch": str(torch.__version__),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "git_commit": git_revision(),
        "manifest_sha256": sha256_file(manifest),
        "note": "Synthetic smoke test; not a Severstal result"
        if config.data_kind == "synthetic"
        else "Baseline; no production deployment claim",
    }
    (output_dir / "run.json").write_text(json.dumps(metadata, indent=2))
    # Copiar o manifesto fixa o split e as anotações usados nesta execução.
    (output_dir / "manifest.json").write_text(Path(manifest).read_text())
    best_score, best_loss, stale, history = -1.0, float("inf"), 0, []
    started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        model.train()
        loss_sum = count = 0
        for image, target, _ in train_batches:
            image, target = image.to(device), target.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = segmentation_loss(model(image), target)
            if not torch.isfinite(loss):
                raise RuntimeError("Loss não finita; verificar entradas e taxa de aprendizado")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach()) * image.shape[0]
            count += image.shape[0]
        val, _ = evaluate_loader(model, val_batches, device, config.threshold)
        score = val["macro_dice"] if val["macro_dice"] is not None else 0.0
        row = {
            "epoch": epoch,
            "train_loss": loss_sum / count,
            "val_loss": val["loss"],
            "val_macro_dice": score,
        }
        history.append(row)
        print(json.dumps(row), flush=True)
        # Val loss desempata Dice=0 no início sem consultar o teste.
        if score > best_score or (score == best_score and val["loss"] < best_loss):
            best_score, best_loss, stale = score, val["loss"], 0
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "config": asdict(config),
                    "epoch": epoch,
                    "val_metrics": val,
                    "manifest_sha256": metadata["manifest_sha256"],
                },
                output_dir / "best.pt",
            )
        else:
            stale += 1
        (output_dir / "history.json").write_text(json.dumps(history, indent=2))
        if stale >= config.patience:
            break
    metadata["elapsed_seconds"] = time.perf_counter() - started
    metadata["best_val_macro_dice"] = best_score
    (output_dir / "run.json").write_text(json.dumps(metadata, indent=2))
    plot_history(history, output_dir / "learning_curves.png")
    return output_dir / "best.pt"


def load_checkpoint(checkpoint_path, device="auto"):
    device = choose_device(device)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    config = Config(**checkpoint["config"])
    model = UNet(config.base_channels)
    model.load_state_dict(checkpoint["model_state"])
    return model.to(device).eval(), config, checkpoint, device


@torch.inference_mode()
def evaluate(data_dir, manifest, checkpoint_path, output_dir, split="test", device="auto"):
    output_dir = Path(output_dir)
    if (output_dir / "metrics.json").exists():
        raise FileExistsError("Avaliação já existe; preserve o teste reservado")
    model, config, checkpoint, selected_device = load_checkpoint(checkpoint_path, device)
    if sha256_file(manifest) != checkpoint["manifest_sha256"]:
        raise ValueError("O manifesto difere daquele usado no treinamento")
    batches = loader(data_dir, manifest, split, config)
    summary, rows = evaluate_loader(model, batches, selected_device, config.threshold)
    summary.update(
        {
            "split": split,
            "threshold": config.threshold,
            "data_kind": config.data_kind,
            "checkpoint_sha256": sha256_file(checkpoint_path),
            "evaluation_resolution": [config.height, config.width],
            "best_epoch": checkpoint["epoch"],
        }
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "metrics.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
    with (output_dir / "per_image_class.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    # Apenas erros efetivos, evitando mostrar vazios corretos como falhas.
    failures = sorted(
        [r for r in rows if r["fp_pixels"] + r["fn_pixels"] > 0],
        key=lambda r: (r["dice"], -r["fn_pixels"] - r["fp_pixels"]),
    )
    (output_dir / "worst_errors.json").write_text(json.dumps(failures[:30], indent=2))
    dataset = batches.dataset
    by_name = {r["image_id"]: i for i, r in enumerate(dataset.records)}
    chosen = list(dict.fromkeys(r["image_id"] for r in failures))[:6]
    if not chosen:
        chosen = [r["image_id"] for r in dataset.records[:3]]
    for n, name in enumerate(chosen):
        image, target, _ = dataset[by_name[name]]
        probabilities = model(image.unsqueeze(0).to(selected_device)).sigmoid()[0].cpu().numpy()
        save_comparison(
            image.numpy().transpose(1, 2, 0),
            target.numpy(),
            probabilities,
            output_dir / f"prediction_{n:02d}.png",
            f"{config.data_kind} | {split} | {name}",
            config.threshold,
        )
    lines = [
        "# Relatório de avaliação",
        "",
        f"Dados: **{config.data_kind}**. Divisão: **{split}**.",
        "",
        "Resultados em resolução de avaliação reduzida; não equivalem ao placar Kaggle.",
        "",
        "| Classe | Dice global | IoU | Recall por pixel | Dice imagens positivas |",
        "|---|---:|---:|---:|---:|",
    ]

    def fmt(x):
        return "n/a" if x is None else f"{x:.4f}"

    for item in summary["classes"]:
        lines.append(
            f"| {item['class_id']} | {fmt(item['dice'])} | {fmt(item['iou'])} | "
            f"{fmt(item['pixel_recall'])} | {fmt(item['positive_image_mean_dice'])} |"
        )
    lines += [
        "",
        "O checkpoint foi selecionado apenas na validação. Limiar fixo em 0,5.",
        "Máscaras vazias corretas não elevam o Dice global por classe.",
        "Consulte per_image_class.csv, worst_errors.json e prediction_*.png.",
    ]
    if config.data_kind == "synthetic":
        lines += [
            "",
            "**DEMONSTRAÇÃO SINTÉTICA: estas métricas não medem desempenho em aço real.**",
        ]
    (output_dir / "report.md").write_text("\n".join(lines), encoding="utf-8")
    return summary


@torch.inference_mode()
def predict(image_path, checkpoint_path, output_dir, device="auto"):
    from PIL import Image

    from .data import encode_rle

    model, config, _, device = load_checkpoint(checkpoint_path, device)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with Image.open(image_path) as image:
        original_size = image.size
        resized = image.convert("RGB").resize(
            (config.width, config.height), Image.Resampling.BILINEAR
        )
    array = np.asarray(resized, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array.transpose(2, 0, 1).copy()).unsqueeze(0).to(device)
    logits = model(tensor)
    # Restaurar probabilidades à resolução original antes de aplicar limiar.
    probabilities = torch.nn.functional.interpolate(
        logits.sigmoid(),
        size=(original_size[1], original_size[0]),
        mode="bilinear",
        align_corners=False,
    )
    masks = probabilities[0].cpu().numpy() >= config.threshold
    np.savez_compressed(output_dir / "prediction.npz", masks=masks)
    payload = {
        "image_id": Path(image_path).name,
        "width": original_size[0],
        "height": original_size[1],
        "threshold": config.threshold,
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "classes": [{"class_id": k + 1, "encoded_pixels": encode_rle(masks[k])} for k in range(4)],
    }
    (output_dir / "prediction.json").write_text(json.dumps(payload, indent=2))
    display_prob = logits.sigmoid()[0].cpu().numpy()
    save_comparison(
        array,
        None,
        display_prob,
        output_dir / "prediction.png",
        "Inferência: sem anotação de referência",
        config.threshold,
    )
    return payload
'''
(PACKAGE / 'engine.py').write_text(source, encoding='utf-8')

### `metrics.py`

In [ ]:
source = r'''"""Métricas globais por classe e Dice separado em imagens positivas."""

import numpy as np


def ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else None


class SegmentationMetrics:
    def __init__(self):
        self.tp = np.zeros(4, dtype=np.int64)
        self.fp = self.tp.copy()
        self.fn = self.tp.copy()
        self.positive_dice = [[] for _ in range(4)]
        self.negative_images = self.tp.copy()
        self.false_positive_images = self.tp.copy()
        self.positive_images = self.tp.copy()
        self.detected_positive_images = self.tp.copy()
        self.rows = []

    def update(self, probabilities, target, image_ids, threshold=0.5):
        pred = np.asarray(probabilities) >= threshold
        true = np.asarray(target) > 0
        if pred.shape != true.shape or pred.ndim != 4 or pred.shape[1] != 4:
            raise ValueError("Esperado [batch, 4, altura, largura] para predição e alvo")
        for i, image_id in enumerate(image_ids):
            for k in range(4):
                p, t = pred[i, k], true[i, k]
                tp, fp, fn = int((p & t).sum()), int((p & ~t).sum()), int((~p & t).sum())
                self.tp[k] += tp
                self.fp[k] += fp
                self.fn[k] += fn
                denom = 2 * tp + fp + fn
                dice = 2 * tp / denom if denom else 1.0
                if t.any():
                    self.positive_dice[k].append(dice)
                    self.positive_images[k] += 1
                    self.detected_positive_images[k] += int(p.any())
                else:
                    self.negative_images[k] += 1
                    self.false_positive_images[k] += int(p.any())
                self.rows.append(
                    {
                        "image_id": image_id,
                        "class_id": k + 1,
                        "dice": dice,
                        "tp_pixels": tp,
                        "fp_pixels": fp,
                        "fn_pixels": fn,
                        "true_positive_pixels": int(t.sum()),
                    }
                )

    def summary(self):
        classes = []
        for k in range(4):
            tp, fp, fn = int(self.tp[k]), int(self.fp[k]), int(self.fn[k])
            classes.append(
                {
                    "class_id": k + 1,
                    "dice": ratio(2 * tp, 2 * tp + fp + fn),
                    "iou": ratio(tp, tp + fp + fn),
                    "pixel_precision": ratio(tp, tp + fp),
                    "pixel_recall": ratio(tp, tp + fn),
                    "positive_image_mean_dice": float(np.mean(self.positive_dice[k]))
                    if self.positive_dice[k]
                    else None,
                    "positive_images": int(self.positive_images[k]),
                    "image_presence_recall": ratio(
                        self.detected_positive_images[k], self.positive_images[k]
                    ),
                    "negative_images": int(self.negative_images[k]),
                    "negative_image_fp_rate": ratio(
                        self.false_positive_images[k], self.negative_images[k]
                    ),
                    "tp_pixels": tp,
                    "fp_pixels": fp,
                    "fn_pixels": fn,
                }
            )
        valid = [c["dice"] for c in classes if c["dice"] is not None]
        return {
            "classes": classes,
            "macro_dice": float(np.mean(valid)) if valid else None,
            "valid_classes": len(valid),
            "conventions": "global pixel counts; absent global class -> null; "
            "per-image empty/empty Dice=1; positive-image Dice excludes empty targets; "
            "image presence means at least one predicted positive pixel",
        }
'''
(PACKAGE / 'metrics.py').write_text(source, encoding='utf-8')

### `model.py`

In [ ]:
source = r'''"""U-Net compacta treinada do zero; quatro canais sigmoid independentes."""

import torch
from torch import nn
from torch.nn import functional as F


def block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.GroupNorm(4, out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.GroupNorm(4, out_channels),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        b = base_channels
        if b < 4 or b % 4:
            raise ValueError("base_channels deve ser múltiplo de 4")
        self.encoders = nn.ModuleList([block(3, b), block(b, b * 2), block(b * 2, b * 4)])
        self.bottleneck = block(b * 4, b * 8)
        self.decoders = nn.ModuleList([block(b * 12, b * 4), block(b * 6, b * 2), block(b * 3, b)])
        self.head = nn.Conv2d(b, 4, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.bottleneck(x)
        for decoder, skip in zip(self.decoders, reversed(skips)):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = decoder(torch.cat([x, skip], dim=1))
        return self.head(x)


def segmentation_loss(logits, target):
    """BCE + soft Dice; sigmoid não deve ser aplicada antes da BCEWithLogits."""
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = logits.sigmoid()
    dims = (0, 2, 3)
    dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
    return bce + 1 - dice.mean()
'''
(PACKAGE / 'model.py').write_text(source, encoding='utf-8')

### `synthetic.py`

In [ ]:
source = r'''"""Gera imagens geométricas para testar execução; não simula aço real."""

import csv
from pathlib import Path

import numpy as np
from PIL import Image

from .data import encode_rle


def generate_synthetic(output_dir, count=40, seed=42):
    output_dir = Path(output_dir)
    if (output_dir / "train.csv").exists():
        raise FileExistsError("Dataset já existe; escolha outro diretório")
    if count < 12:
        raise ValueError("Use pelo menos 12 exemplos sintéticos")
    image_dir = output_dir / "train_images"
    image_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    height, width = 64, 256
    rows = []
    colors = [(220, 60, 60), (60, 220, 60), (60, 60, 220), (230, 200, 50)]
    for i in range(count):
        image = rng.integers(35, 70, (height, width, 3), dtype=np.uint8)
        masks = np.zeros((4, height, width), dtype=np.uint8)
        # 20% de imagens normais; algumas imagens multiclasse.
        selected = [] if i % 5 == 0 else [i % 4]
        if i % 7 == 0 and selected:
            selected.append((i + 1) % 4)
        for k in selected:
            top, left = int(rng.integers(5, 35)), int(rng.integers(5, 210))
            masks[k, top : top + 15, left : left + 30] = 1
            image[masks[k] > 0] = colors[k]
        name = f"synthetic_{i:04d}.png"
        Image.fromarray(image).save(image_dir / name)
        for k in range(4):
            rows.append({"ImageId": name, "ClassId": k + 1, "EncodedPixels": encode_rle(masks[k])})
    with (output_dir / "train.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["ImageId", "ClassId", "EncodedPixels"])
        writer.writeheader()
        writer.writerows(rows)
    (output_dir / "SYNTHETIC.txt").write_text(
        "Dados geométricos artificiais para smoke test. Não são imagens Severstal ou aço real.\n"
    )
    return output_dir
'''
(PACKAGE / 'synthetic.py').write_text(source, encoding='utf-8')

### `visualize.py`

In [ ]:
source = r'''"""Análise exploratória, curvas e comparação visual de máscaras."""

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

COLORS = np.array([[1.0, 0.75, 0.15], [0.7, 0.35, 1.0], [0.1, 0.85, 0.8], [1.0, 0.25, 0.4]])


def overlay(image, masks):
    result = image.copy()
    for channel, color in zip(masks, COLORS):
        selected = channel > 0
        result[selected] = 0.55 * result[selected] + 0.45 * color
    return np.clip(result, 0, 1)


def save_comparison(image, target, probabilities, output, title, threshold=0.5):
    arrays, labels = [image], ["Imagem"]
    if target is not None:
        arrays.append(overlay(image, target))
        labels.append("Máscara real")
    if probabilities is not None:
        arrays.append(overlay(image, probabilities >= threshold))
        labels.append("Predição")
    fig, axes = plt.subplots(1, len(arrays), figsize=(5 * len(arrays), 4), constrained_layout=True)
    for ax, array, label in zip(np.atleast_1d(axes), arrays, labels):
        ax.imshow(array)
        ax.set_title(label)
        ax.axis("off")
    fig.suptitle(title + " | classes: 1 dourado · 2 roxo · 3 ciano · 4 rosa", fontsize=10)
    fig.savefig(output, dpi=140)
    plt.close(fig)


def explore(data_dir, manifest_path, output_dir, size=(128, 800)):
    from .data import SteelDataset

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest = json.loads(Path(manifest_path).read_text())
    records = manifest["records"]
    summary = {
        "images": len(records),
        "unique_file_hashes": len({r["sha256"] for r in records}),
        "splits": {},
        "resolution_counts": {},
    }
    for record in records:
        key = f"{record['width']}x{record['height']}"
        summary["resolution_counts"][key] = summary["resolution_counts"].get(key, 0) + 1
    for split in ["train", "val", "test"]:
        selected = [r for r in records if r["split"] == split]
        summary["splits"][split] = {
            "images": len(selected),
            "no_defect": sum(not any(r["rles"]) for r in selected),
            "positive_images_per_class": {
                str(k + 1): sum(bool(r["rles"][k]) for r in selected) for k in range(4)
            },
        }
    (output_dir / "data_summary.json").write_text(json.dumps(summary, indent=2))
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    for j, split in enumerate(["train", "val", "test"]):
        counts = summary["splits"][split]["positive_images_per_class"]
        ax.bar(np.arange(4) + j * 0.25, [counts[str(k)] for k in range(1, 5)], 0.25, label=split)
    ax.set_xticks(np.arange(4) + 0.25, ["Classe 1", "Classe 2", "Classe 3", "Classe 4"])
    ax.set_ylabel("Imagens positivas (classes podem coexistir)")
    ax.legend()
    fig.savefig(output_dir / "class_distribution.png", dpi=140)
    plt.close(fig)
    dataset = SteelDataset(data_dir, manifest_path, "train", size=size)
    # EDA visual usa apenas treino; os exemplos incluem positivos e negativos quando disponíveis.
    candidates = sorted(
        range(len(dataset)), key=lambda i: -sum(bool(v) for v in dataset.records[i]["rles"])
    )
    chosen = candidates[:3]
    negatives = [i for i in candidates if not any(dataset.records[i]["rles"])]
    if negatives and negatives[0] not in chosen:
        chosen.append(negatives[0])
    for i in chosen:
        image, mask, name = dataset[i]
        save_comparison(
            image.numpy().transpose(1, 2, 0),
            mask.numpy(),
            None,
            output_dir / f"example_{i}.png",
            f"EDA treino: {name}",
        )
    return summary


def plot_history(history, output):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    axes[0].plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="treino")
    axes[0].plot([r["epoch"] for r in history], [r["val_loss"] for r in history], label="validação")
    axes[0].set_title("BCE + soft Dice")
    axes[0].legend()
    axes[1].plot([r["epoch"] for r in history], [r["val_macro_dice"] for r in history])
    axes[1].set_title("Macro Dice de validação · limiar fixo 0,5")
    for ax in axes:
        ax.set_xlabel("Época")
    fig.savefig(output, dpi=140)
    plt.close(fig)
'''
(PACKAGE / 'visualize.py').write_text(source, encoding='utf-8')

## 3. Localização e validação dos dados

In [ ]:
import json
import sys
import torch
import numpy as np
from PIL import Image as PILImage

sys.path.insert(0, str(PACKAGE_ROOT))
from steelvision.data import prepare_manifest
from steelvision.engine import Config, train, evaluate, predict, choose_device
from steelvision.synthetic import generate_synthetic
from steelvision.visualize import explore

if USE_SYNTHETIC:
    DATA_DIR = generate_synthetic(OUTPUT / "synthetic_data", seed=SEED)
    torch.set_num_threads(2)
    SIZE, CHANNELS, DEVICE = (64, 256), 8, "cpu"
else:
    if DATA_DIR_OVERRIDE is not None:
        DATA_DIR = Path(DATA_DIR_OVERRIDE)
    else:
        candidates = sorted(p.parent for p in Path("/kaggle/input").rglob("train.csv")
                            if (p.parent / "train_images").is_dir())
        if len(candidates) != 1:
            raise RuntimeError("Adicione os dados Severstal ao notebook ou configure DATA_DIR_OVERRIDE. "
                               f"Encontrados {len(candidates)} datasets compatíveis.")
        DATA_DIR = candidates[0]
    SIZE, CHANNELS, DEVICE = (128, 800), 16, "auto"

manifest = prepare_manifest(DATA_DIR, OUTPUT / "prepared", seed=SEED)
print("Dados:", DATA_DIR)
print("Modo:", "SINTÉTICO — não usar como métrica Severstal" if USE_SYNTHETIC else "Severstal")
print("Dispositivo:", choose_device(DEVICE))
print("PyTorch:", torch.__version__)
print("Manifesto:", manifest)


## 4. Exploração das classes e máscaras
A divisão ocorre antes dos aumentos. Duplicatas exatas por hash ficam no mesmo conjunto.
Sem identificadores de bobina/lote, isso não garante independência industrial. A divisão não é
estratificada; confira se há exemplos suficientes por classe em validação e teste.
As imagens normais são preservadas, inclusive quando não possuem linhas positivas no CSV.


In [ ]:
summary = explore(DATA_DIR, manifest, OUTPUT / "eda", size=SIZE)
print(json.dumps(summary, indent=2, ensure_ascii=False))

def show_file(path, figsize=None):
    with PILImage.open(path) as image:
        display(image.copy())

show_file(OUTPUT / "eda" / "class_distribution.png", (9, 5))
for path in sorted((OUTPUT / "eda").glob("example_*.png"))[:3]:
    show_file(path)


## 5. Treinamento da U-Net
Baseline compacto treinado do zero, sem pesos externos. Os quatro canais são independentes:
as classes podem coexistir. A loss combina BCEWithLogits e soft Dice.

Imagem: bilinear. Máscara: nearest-neighbor. Aumentos sincronizados apenas no treino.
A resolução 128 × 800 reduz custo, mas pode apagar defeitos pequenos; métricas são calculadas nessa resolução.
O melhor checkpoint é selecionado por macro Dice de validação, com desempate pela loss.


In [ ]:
config = Config(epochs=EPOCHS, batch_size=BATCH_SIZE, height=SIZE[0], width=SIZE[1],
                base_channels=CHANNELS, seed=SEED, device=DEVICE,
                data_kind="synthetic" if USE_SYNTHETIC else "severstal")
checkpoint = train(DATA_DIR, manifest, OUTPUT / "run", config)
show_file(OUTPUT / "run" / "learning_curves.png", (11, 5))
print("Checkpoint:", checkpoint)


## 6. Avaliação final no teste reservado
Execute após congelar o experimento. Não use estas métricas para ajustar o modelo.
Para estudar erros durante desenvolvimento, use `evaluate(..., split="val")` em outro diretório.

O Dice global por classe não é inflado por máscaras vazias corretas. Há também Dice apenas em imagens
positivas, recall por pixel e taxa de falso positivo em imagens negativas. Classes ausentes recebem `null`
quando a métrica não é definida. Esses números não reproduzem o placar oficial Kaggle.


In [ ]:
metrics = evaluate(DATA_DIR, manifest, checkpoint, OUTPUT / "test", device=DEVICE)
print((OUTPUT / "test" / "report.md").read_text())
print(json.dumps(metrics, indent=2, ensure_ascii=False))
for path in sorted((OUTPUT / "test").glob("prediction_*.png"))[:4]:
    show_file(path)


## 7. Inferência em uma imagem
O exemplo usa uma imagem do próprio teste reservado. A predição não modifica o checkpoint.
As probabilidades são restauradas às dimensões originais antes do limiar; a saída é salva em NPZ e RLE.
Não há máscara de referência no painel de inferência isolada.


In [ ]:
records = json.loads(Path(manifest).read_text())["records"]
example = next(record["image_id"] for record in records if record["split"] == "test")
result = predict(DATA_DIR / "train_images" / example, checkpoint, OUTPUT / "inference", DEVICE)
print("Imagem:", result["image_id"], "| dimensões:", result["height"], result["width"])
show_file(OUTPUT / "inference" / "prediction.png")


## 8. Salvar os resultados e próximos experimentos
O ZIP abaixo contém os artefatos da execução e o checkpoint, sem redistribuir as imagens de treino.
Use o painel de arquivos de saída do Kaggle para baixar.

Próximas hipóteses, avaliadas na validação: maior resolução, encoder pré-treinado, amostragem balanceada,
pesos por classe e limiares por classe. Congele as decisões antes de uma nova avaliação final independente.
ONNX, API de inferência, MLflow e monitoramento de drift são extensões posteriores.


In [ ]:
import zipfile

archive = BASE / f"steelvision_{RUN_NAME}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and "synthetic_data" not in path.relative_to(OUTPUT).parts:
            bundle.write(path, path.relative_to(OUTPUT))
print("Artefatos disponíveis:", archive)
